## 1. Environment Setup

In [1]:
from dotenv import dotenv_values
import os
import sys
import warnings

# HPC-specific paths for Arrow and FAISS — adjust or remove for local environments
sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/arrow/24.0.0/lib/python3.12/site-packages')
sys.path.insert(0, '/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/CUDA/gcc12/cuda12.6/faiss/1.12.0/lib/python3.12/site-packages')

warnings.filterwarnings('ignore')

config = dotenv_values(".env")
os.environ["OPENROUTER_API_KEY"] = config['OPENROUTER_API_KEY']
os.environ["HUGGINGFACEHUB_API_TOKEN"] = config['HF_API_KEY']
OPENROUTER_API_KEY = config['OPENROUTER_API_KEY']

print("Environment loaded.")

Environment loaded.


## 2. Vector Database (RAG Component)

We build a **FAISS** index over the three PDFs using `all-MiniLM-L6-v2` embeddings — a lightweight but effective sentence transformer.

**Chunking strategy:**
- `chunk_size=1500` characters — large enough to preserve argument context across sentences
- `chunk_overlap=200` — prevents losing information at chunk boundaries
- Retriever returns top `k=6` chunks per query

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

PDF_PATHS = [
    'data/2007_badre.pdf',      # Badre & D'Esposito — hierarchical PFC framework
    'data/2023_multitask.pdf',  # Ito et al. — MDTB 26-task fMRI dataset
    'data/2024_demand.pdf',     # Assem et al. — multiple-demand executive tasks
]

# Load and split
docs = []
for pdf in PDF_PATHS:
    loaded = PyPDFLoader(pdf).load()
    docs.extend(loaded)
    print(f"  Loaded {len(loaded)} pages from {pdf}")

splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
chunks = splitter.split_documents(docs)
print(f"\nTotal chunks: {len(chunks)}")

# Embed and index
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 6})

print("FAISS index built successfully.")

  Loaded 18 pages from data/2007_badre.pdf
  Loaded 28 pages from data/2023_multitask.pdf
  Loaded 19 pages from data/2024_demand.pdf

Total chunks: 258


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index built successfully.


In [3]:
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


GENERATOR_MODEL  = "nvidia/nemotron-3-nano-30b-a3b:free"
CRITIC_MODEL     = "nvidia/nemotron-3-nano-30b-a3b:free"
 
generator_chat_llm = ChatOpenAI(
    model=GENERATOR_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.3,        # Low temp → more factual, grounded questions
    max_tokens=1024,
)
 
critic_chat_llm = ChatOpenAI(
    model=CRITIC_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.1,        # Very low temp → conservative quality filtering
    max_tokens=512,
)
 
# Local embeddings — identical model used in your FAISS index,
# so similarity scores are comparable across retrieval and evaluation.
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
 
# Wrap for RAGAS
generator_llm  = LangchainLLMWrapper(generator_chat_llm)
critic_llm     = LangchainLLMWrapper(critic_chat_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings_model)
 
print("LLM and embeddings configured.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LLM and embeddings configured.


In [ ]:
# from ragas.llms import LangchainLLMWrapper
# print([m for m in dir(LangchainLLMWrapper) if 'generate' in m.lower()])

In [ ]:
# import nest_asyncio
# nest_asyncio.apply()

# from ragas.llms import LangchainLLMWrapper

# async def _agenerate_prompt(self, prompts, stop=None, callbacks=None, **kwargs):
#     messages_list = [p.to_messages() for p in prompts]
#     # try agenerate first, fall back to agenerate_messages
#     fn = getattr(self.langchain_llm, "agenerate", None) or \
#          getattr(self.langchain_llm, "agenerate_messages", None)
#     return await fn(messages_list, stop=stop, callbacks=callbacks, **kwargs)

# def _generate_prompt(self, prompts, stop=None, callbacks=None, **kwargs):
#     messages_list = [p.to_messages() for p in prompts]
#     fn = getattr(self.langchain_llm, "generate", None) or \
#          getattr(self.langchain_llm, "generate_messages", None)
#     return fn(messages_list, stop=stop, callbacks=callbacks, **kwargs)

# LangchainLLMWrapper.agenerate_prompt = _agenerate_prompt
# LangchainLLMWrapper.generate_prompt  = _generate_prompt

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Synthetic Testset Generation
#
#    RAGAS generates three types of questions via "evolutions":
#
#    simple       → directly answerable from a single chunk
#                   e.g. "What 4 hierarchy levels does Badre 2007 define?"
#
#    reasoning    → requires multi-hop inference within the retrieved context
#                   e.g. "If feature competition activates pre-PMd, which region
#                         handles a higher abstraction level?"
#
#    multi_context → answer requires combining information from 2+ chunks
#                   e.g. "How does Assem 2024's n-back finding relate to
#                         Badre 2007's context level prediction?"
#
#    Distribution choice rationale for a scientific meta-analysis RAG:
#      - 40% simple:       validates basic retrieval (does the right chunk surface?)
#      - 40% reasoning:    validates that agents reason correctly over retrieved text
#      - 20% multi_context: validates cross-paper synthesis — the core hard problem
#      - 0%  conditional:  conditional questions ("if X then Y") don't map well to
#                          the neuroscience domain and tend to produce ill-formed
#                          questions from free models; excluded deliberately.
# ─────────────────────────────────────────────────────────────────────────────
# from ragas.testset.generator  import TestsetGenerator
from ragas.testset import TestsetGenerator
 
generator = TestsetGenerator.from_langchain(
    generator_llm,
    critic_llm,
    ragas_embeddings,
)
 
print("\nGenerating synthetic testset — this makes multiple LLM calls...")
print("Expected time: 3–8 minutes for 15 questions on free-tier rate limits.\n")


Generating synthetic testset — this makes multiple LLM calls...
Expected time: 3–8 minutes for 15 questions on free-tier rate limits.



In [ ]:
distributions = {
    "simple":        0.4,
    "reasoning":     0.4,
    "multi_context": 0.2,
}

testset = generator.generate_with_langchain_docs(
    documents           = chunks,
    testset_size        = 15,        
    query_distribution  = distributions,
    with_debugging_logs = False,
)

 
# Convert to pandas for inspection and storage
test_df = testset.to_pandas()
 
print(f"\nGenerated {len(test_df)} test cases.")
print(f"Columns: {list(test_df.columns)}")
print(f"\nQuestion types distribution:\n{test_df['evolution_type'].value_counts()}")
print("\nSample questions:")

Applying SummaryExtractor:   0%|          | 0/249 [00:00<?, ?it/s]

Task failed with AttributeError: 'LangchainLLMWrapper' object has no attribute 'agenerate_prompt'
Task failed with AttributeError: 'LangchainLLMWrapper' object has no attribute 'agenerate_prompt'
Task failed with AttributeError: 'LangchainLLMWrapper' object has no attribute 'agenerate_prompt'
Task failed with AttributeError: 'LangchainLLMWrapper' object has no attribute 'agenerate_prompt'
Task failed with AttributeError: 'LangchainLLMWrapper' object has no attribute 'agenerate_prompt'
Task failed with AttributeError: 'LangchainLLMWrapper' object has no attribute 'agenerate_prompt'
Task failed with AttributeError: 'LangchainLLMWrapper' object has no attribute 'agenerate_prompt'
Task failed with AttributeError: 'LangchainLLMWrapper' object has no attribute 'agenerate_prompt'
Task failed with AttributeError: 'LangchainLLMWrapper' object has no attribute 'agenerate_prompt'
Task failed with AttributeError: 'LangchainLLMWrapper' object has no attribute 'agenerate_prompt'
Task failed with Att

In [ ]:
import inspect 

print(inspect.signature(generator.generate_with_langchain_docs))

In [ ]:
for _, row in test_df.head(3).iterrows():
    print(f"  [{row['evolution_type']}] {row['question'][:120]}...")
 
# Save — this is your reusable ground truth dataset.
# Re-run generation only if you change your corpus or chunking strategy.
test_df.to_csv("synthetic_ground_truth.csv", index=False)
print("\nSaved to synthetic_ground_truth.csv")

## 4. Tools

### 4a. RAG Tool — `paper_search`

Queries the local FAISS index and returns the top-k chunks with source metadata. This is the primary retrieval mechanism for all three agents.

In [ ]:

@tool("paper_search")
def paper_search(query: str) -> str:
    """
    Search the uploaded research papers for passages relevant to the query.
    Papers available: Badre & D'Esposito (2007) on PFC hierarchy, Ito et al. (2023)
    on multitask fMRI, and Assem et al. (2024) on multiple-demand executive tasks.
    Returns the top 6 most relevant text chunks with source file labels.
    Use specific, targeted queries (e.g., 'PFC activation n-back task Ito 2023') 
    rather than broad ones for best results.
    """
    retrieved_docs = retriever.invoke(query)
    if not retrieved_docs:
        return "No relevant passages found. Try rephrasing your query."
    return "\n\n---\n\n".join([
        f"[Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in retrieved_docs
    ])

### 4b. Web Tool — `biorxiv_search`

Scrapes bioRxiv search results to check whether the findings match recent preprint literature. Includes error handling and graceful fallback.

In [ ]:
import requests
from bs4 import BeautifulSoup

@tool("biorxiv_search")
def biorxiv_search(query: str) -> str:
    """
    Search bioRxiv for preprints relevant to the query.
    Use this to check whether findings in the local papers are corroborated 
    by recent neuroscience preprints. Returns titles and abstracts of up to 
    5 results. If no results are found or the search fails, returns a specific 
    explanation so the agent can reason about the absence of evidence.
    """
    try:
        url = f"https://www.biorxiv.org/search/{query.replace(' ', '%20')}"
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        results = []
        for article in soup.select(".highwire-article-citation")[:5]:
            title = article.select_one(".highwire-cite-title")
            abstract = article.select_one(".highwire-cite-snippet")
            if title:
                results.append(
                    f"Title: {title.get_text(strip=True)}\n"
                    f"Snippet: {abstract.get_text(strip=True) if abstract else 'No abstract available'}"
                )

        if results:
            return f"Found {len(results)} result(s) on bioRxiv:\n\n" + "\n\n".join(results)
        else:
            return (
                f"No preprints found on bioRxiv for query: '{query}'. "
                "This may indicate the topic is primarily covered in peer-reviewed journals, "
                "or that the query terms need adjustment. Consider rephrasing or checking "
                "PubMed for published literature."
            )

    except requests.exceptions.Timeout:
        return "bioRxiv search timed out. The site may be temporarily unavailable."
    except requests.exceptions.RequestException as e:
        return f"bioRxiv search failed with network error: {str(e)}. Proceed using local paper evidence only."
    except Exception as e:
        return f"Unexpected error during bioRxiv search: {str(e)}."

## 5. Agent Definitions

Each agent has a distinct scientific role. **Roles** describe professional identity; **goals** describe measurable success; **backstories** provide domain context that shapes how the LLM reasons.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Agent 1: Task Classification Specialist
# Responsibility: Extract and classify tasks by abstraction level
# ─────────────────────────────────────────────────────────────
task_classification_agent = Agent(
    role="Cognitive Task Classification Specialist",
    goal=(
        "Produce a complete, evidence-grounded classification of every cognitive task "
        "in the Ito (2023) and Assem (2024) papers as either abstract or concrete, "
        "using the operational definition provided by Badre & D'Esposito (2007). "
        "Every classification must cite specific retrieved text as justification."
    ),
    backstory=(
        "You are a cognitive neuroscientist with expertise in task taxonomy and "
        "hierarchical theories of cognitive control. You are meticulous about "
        "distinguishing tasks that require abstract rule selection (e.g., task-switching, "
        "working memory updating) from those that operate on concrete stimulus-response "
        "mappings (e.g., simple motor responses, passive viewing). You never classify "
        "a task without first retrieving its description from the papers."
    ),
    tools=[paper_search],
    allow_delegation=False,
    llm=llm,
    verbose=True
)

# ─────────────────────────────────────────────────────────────
# Agent 2: Neuroimaging Data Analyst
# Responsibility: Extract PFC activation findings per task from the papers
# ─────────────────────────────────────────────────────────────
activation_analyst_agent = Agent(
    role="Neuroimaging Data Analyst",
    goal=(
        "For each task in the Ito (2023) and Assem (2024) papers, retrieve and report "
        "the specific PFC subregions (e.g., frontopolar, DLPFC, premotor cortex, PMd, "
        "pre-PMd, IFJ) that showed significant activation, citing page numbers or "
        "figure references. Do NOT infer activation patterns — only report what is "
        "explicitly stated in retrieved text."
    ),
    backstory=(
        "You are an fMRI data analyst specializing in frontal lobe functional organization. "
        "You are trained to distinguish between frontopolar cortex (BA10), dorsolateral PFC "
        "(DLPFC, BA46/9), inferior frontal junction (IFJ), premotor cortex (PMd/PMv), "
        "and primary motor cortex (M1) — and you know that these regions span anterior-to-posterior "
        "PFC. You are rigorous about not extrapolating beyond what the data shows: if a paper "
        "does not report task-specific PFC subregion activation, you report that as a data gap "
        "rather than inventing a result."
    ),
    tools=[paper_search],
    allow_delegation=False,
    llm=llm,
    verbose=True
)

# ─────────────────────────────────────────────────────────────
# Agent 3: Scientific Claims Verifier
# Responsibility: Cross-check findings against literature and flag unsupported claims
# ─────────────────────────────────────────────────────────────
scientific_verifier_agent = Agent(
    role="Scientific Claims Verifier",
    goal=(
        "Evaluate whether the evidence from Agents 1 and 2 is sufficient to support "
        "the claim that more abstract tasks activate more anterior PFC. "
        "Flag any gaps, inconsistencies, or unsupported inferences. "
        "Search bioRxiv for corroborating or contradicting recent literature. "
        "Deliver a calibrated verdict: supported / partially supported / not supported, "
        "with explicit reasoning."
    ),
    backstory=(
        "You are a scientific peer reviewer with expertise in meta-analysis and "
        "replication science in cognitive neuroscience. You have reviewed manuscripts "
        "on prefrontal cortex organization for journals including Nature Neuroscience "
        "and Cerebral Cortex. You are skeptical of overinterpretation and always "
        "distinguish between 'the paper shows X' and 'the available evidence suggests X.' "
        "You note when a dataset (e.g., MDTB) was not designed to test anterior-posterior "
        "gradients explicitly, and flag when conclusions go beyond what the data support."
    ),
    tools=[paper_search, biorxiv_search],
    allow_delegation=False,
    llm=llm,
    verbose=True
)

print("Agents defined.")

## 6. Task Definitions

Tasks specify *what to do*, *what format to produce*, and *which agent is responsible*. Clear expected outputs constrain the LLM and make outputs programmatically parseable.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 1: Retrieve the abstract/concrete framework + classify all tasks
# ─────────────────────────────────────────────────────────────
classify_tasks = Task(
    description=(
        "Complete the following steps in order:\n\n"
        "**Step 1 — Retrieve the definition of abstract vs. concrete tasks.**\n"
        "Use paper_search with the query: 'Badre D\'Esposito hierarchical abstraction "
        "definition concrete abstract levels representation'. "
        "Extract the operational definition: what makes a task abstract vs. concrete "
        "according to the hierarchical framework (e.g., the level at which competing "
        "representations must be resolved).\n\n"
        "**Step 2 — Retrieve the task list from Ito et al. (2023).**\n"
        "Use paper_search with: 'MDTB 26 tasks list multitask fMRI Ito 2023'. "
        "Extract the full list of tasks from the MDTB dataset.\n\n"
        "**Step 3 — Retrieve the task list from Assem et al. (2024).**\n"
        "Use paper_search with: 'Assem 2024 executive tasks n-back switch stop signal'. "
        "Extract the tasks used and their easy/hard conditions.\n\n"
        "**Step 4 — Classify each task.**\n"
        "Apply the Badre & D'Esposito definition to classify each task as Abstract or "
        "Concrete. For each task, state which level of the hierarchy it engages "
        "(response, feature, dimension, or context level) and provide a one-sentence "
        "justification citing evidence from the retrieved text."
    ),
    expected_output=(
        "A markdown table with columns:\n"
        "| Task Name | Paper | Abstract / Concrete | Hierarchy Level | Justification (with text evidence) |\n"
        "Followed by a brief paragraph summarizing the Badre & D'Esposito definition used."
    ),
    agent=task_classification_agent
)

# ─────────────────────────────────────────────────────────────
# Task 2: Retrieve PFC activation data per task
# ─────────────────────────────────────────────────────────────
# analyze_activation = Task(
#     description=(
#         "Using paper_search, retrieve PFC activation findings for tasks in the "
#         "Ito (2023) and Assem (2024) papers. Follow these steps:\n\n"
#         "**Step 1:** Search for 'PFC activation results Ito 2023 multitask frontal cortex' "
#         "and 'prefrontal activation maps multitask representational topography'.\n\n"
#         "**Step 2:** Search for 'Assem 2024 n-back switch stop PFC frontal activation "
#         "results MD regions'.\n\n"
#         "**Step 3:** Search for 'Badre D\'Esposito PMd pre-PMd IFJ frontopolar activation "
#         "response feature dimension context experiments' to get the reference activation "
#         "profile for each hierarchical level.\n\n"
#         "**Important:** Only report activations that are explicitly described in retrieved "
#         "text. If the paper reports whole-brain activation maps without naming specific "
#         "PFC subregions per task, state this explicitly as a data gap rather than inferring."
#     ),
#     expected_output=(
#         "A markdown table with columns:\n"
#         "| Task | Paper | PFC Subregion Activated | Anterior or Posterior | Evidence Quote / Figure Reference |\n\n"
#         "Followed by a 'Data Gaps' section listing any tasks for which PFC subregion "
#         "activation was not explicitly reported in the retrieved text."
#     ),
#     agent=activation_analyst_agent
# )

analyze_activation = Task(
    description=(
        "Retrieve PFC-related findings from Ito (2023) and Assem (2024) only. "
        "Do NOT use Badre (2007) for activation data.\n\n"
        "**Step 1 — Assem (2024) sub-areal frontal findings.**\n"
        "Search: 'Assem 2024 n-back dorsal frontal 8Ad anterior DMN'\n"
        "Search: 'Assem 2024 stop CON switch DAN frontal preference vertices'\n"
        "Search: 'Assem 2024 p9-46v anterior dorsal posterior ventral border'\n"
        "Extract any named frontal areas and which task preferentially activated them.\n\n"
        "**Step 2 — Ito (2023) frontal hierarchy descriptions.**\n"
        "Search: 'Ito 2023 frontal hierarchy representational alignment CPRO n-back'\n"
        "Search: 'Ito 2023 anterior frontal cortex task topography'\n"
        "Search: 'multitask intrinsic hierarchy frontal cortex high demand'\n"
        "Extract any text describing where in frontal cortex tasks are represented.\n\n"
        "**Step 3 — Be explicit about what these papers can and cannot say.**\n"
        "Neither paper was designed to test rostro-caudal abstraction gradients. "
        "State clearly for each paper: (a) what frontal activation language IS present, "
        "(b) what the paper's actual design goal was, and (c) whether the available "
        "text supports any anterior-posterior inference at all."
    ),
    expected_output=(
        "A markdown table with columns:\n"
        "| Task | Paper | Frontal Region Named | Anterior or Posterior | "
        "Evidence Quote / Figure Reference |\n\n"
        "Only populate rows where text explicitly names a frontal region. "
        "Leave no row as N/A without a one-sentence explanation of why.\n\n"
        "'Data Gaps' section must distinguish between:\n"
        "- **Retrieval gap**: text exists but wasn't found\n"
        "- **Design gap**: paper never reported this because it wasn't the study goal\n"
    ),
    agent=activation_analyst_agent
)


# ─────────────────────────────────────────────────────────────
# Task 3: Verify the central claim and situate in literature
# ─────────────────────────────────────────────────────────────
# verify_claim = Task(
#     description=(
#         "Evaluate whether the combined findings from Tasks 1 and 2 support the claim:\n"
#         "'More abstract cognitive tasks activate more anterior PFC regions.'\n\n"
#         "**Step 1 — Assess internal consistency.**\n"
#         "Cross-reference the task classifications (Task 1) with the activation data "
#         "(Task 2). For each task where both classification AND PFC activation data are "
#         "available, does the pattern hold (abstract → anterior; concrete → posterior)? "
#         "Note any exceptions.\n\n"
#         "**Step 2 — Assess data gaps.**\n"
#         "Identify how many tasks from Task 2 had missing PFC activation data. "
#         "How does this affect the strength of the conclusion?\n\n"
#         "**Step 3 — Search recent literature.**\n"
#         "Use biorxiv_search with: 'anterior prefrontal cortex abstract cognitive control hierarchy'. "
#         "Then try: 'rostro-caudal gradient PFC abstraction fMRI'. "
#         "Report what you find, or explicitly note if bioRxiv returns no results and why that "
#         "may be (e.g., topic is mature and primarily covered in journals).\n\n"
#         "**Step 4 — Deliver a verdict.**\n"
#         "Choose one: 'Supported', 'Partially Supported', or 'Not Supported by available data'. "
#         "Justify your verdict with specific reference to the evidence."
#     ),
#     expected_output=(
#         "A structured scientific summary with the following sections:\n"
#         "1. **Evidence For** — tasks where abstract classification + anterior PFC activation align\n"
#         "2. **Evidence Against / Exceptions** — any mismatches\n"
#         "3. **Data Gaps** — tasks without activation data, impact on conclusions\n"
#         "4. **Literature Corroboration** — bioRxiv results or explanation of absence\n"
#         "5. **Verdict** — Supported / Partially Supported / Not Supported + one-paragraph justification\n"
#         "6. **Key References** in Author, Year, Title format"
#     ),
#     agent=scientific_verifier_agent
# )


FRONTAL_ANATOMY_REFERENCE = """
FRONTAL LOBE ANTERIOR-POSTERIOR REFERENCE (Glasser HCP MMP1.0 parcellation):

ANTERIOR PFC (rostral):
- Area 10 / BA10 / frontopolar cortex — most anterior
- Area 9 (dorsomedial PFC, 9m)
- Area 8Ad (anterior part of area 8, borders DMN) — anterior
- Area p9-46v anterior-dorsal border — anterior
- Area 47 / 47l (lateral orbitofrontal) — anterior-ventral

MID-LATERAL PFC:
- Area 46 / 9-46v / p9-46v — mid-lateral DLPFC
- Area 9-46d — mid-lateral DLPFC
- Area 8Av / 8C — mid-frontal

POSTERIOR/PREMOTOR PFC:
- Area 6 / 6a (dorsal premotor, DAN-associated) — posterior frontal
- Area 6r / IFJ (inferior frontal junction) — posterior-lateral
- Area 55b — posterior lateral PFC
- FOP5 (frontal operculum) — posterior-ventral

KEY MAPPINGS FROM ASSEM (2024):
- 3>1 n-back preferring vertices → overlap with 8Ad (ANTERIOR)
- stop>no stop preferring vertices → overlap with CON, area 46, 6r (MID to POSTERIOR)
- switch>no switch preferring vertices → overlap with DAN, area 6a (POSTERIOR)
- p9-46v anterior-dorsal border → stop activations (MID-ANTERIOR)
- p9-46v posterior-ventral border → switch activations (MID-POSTERIOR)
"""

verify_claim = Task(
    description=(
        "Evaluate whether findings from Ito (2023) and Assem (2024) support:\n"
        "'More abstract cognitive tasks activate more anterior PFC regions.'\n\n"
        "IMPORTANT: Use the anatomical reference below to classify any named "
        "frontal region as anterior, mid, or posterior — do NOT wait for the "
        "papers to use the word 'anterior' explicitly.\n\n"
        f"{FRONTAL_ANATOMY_REFERENCE}\n\n"
        "**Step 1 — Map reported regions onto anterior-posterior axis.**\n"
        "From Task 2 output, take every named frontal region and classify it "
        "using the reference above. Then cross-reference with Task 1 "
        "classifications (abstract vs. concrete).\n"
        "Key mappings to apply from Assem (2024):\n"
        "- 3>1 n-back → area 8Ad → ANTERIOR\n"
        "- stop>no stop → area 46, 6r → MID to POSTERIOR\n"
        "- switch>no switch → area 6a → POSTERIOR\n\n"
        "**Step 2 — Test the gradient claim.**\n"
        "Given that n-back is classified as Abstract (context level) and "
        "activates 8Ad (anterior), while switch activates 6a (posterior) — "
        "does this partial pattern support the claim?\n\n"
        "**Step 3 — Search literature.**\n"
        "Search paper_search: 'Assem 2024 8Ad anterior frontal n-back DMN'\n"
        "Search paper_search: 'Ito 2023 frontal hierarchy representational topography'\n\n"
        "**Step 4 — Verdict.**\n"
        "Choose: Supported / Partially Supported / Not Supported.\n"
        "Be specific: cite which region-task pairs drive the verdict."
    ),
    expected_output=(
        "1. **Region Classification Table**\n"
        "| Task | Paper | Region Named | A/P Classification | Abstract? | Gradient Consistent? |\n\n"
        "2. **Evidence For** — specific region-task pairs that fit abstract→anterior\n"
        "3. **Evidence Against** — mismatches\n"
        "4. **Data Gaps** — design gap vs retrieval gap, per paper\n"
        "5. **Verdict** with one-paragraph justification citing specific regions\n"
        "6. **Key References**"
    ),
    agent=scientific_verifier_agent
)


print("Tasks defined.")

## 7. Run the Crew

In [ ]:
import os
import tempfile

# Point CrewAI's SQLite storage to local scratch instead of /project NFS
local_db_dir = os.path.join(tempfile.gettempdir(), "crewai_storage")
os.makedirs(local_db_dir, exist_ok=True)
os.environ["CREWAI_STORAGE_DIR"] = local_db_dir

In [ ]:
crew = Crew(
    agents=[task_classification_agent, activation_analyst_agent, scientific_verifier_agent],
    tasks=[classify_tasks, analyze_activation, verify_claim],
    max_rpm=10,       # Respect free-tier rate limits
    verbose=True,
    memory=False      # No cross-session memory; all context passed via task outputs
)

result = crew.kickoff()

## 8. Results

In [ ]:
from IPython.display import Markdown
Markdown(result.raw)

## 9. Retrieval Quality Audit

Inspect which chunks were actually retrieved and used — important for debugging RAG hallucinations and validating that the correct papers were consulted.

In [ ]:
def audit_retrieval(query: str, top_k: int = 3) -> None:
    """Display the top-k retrieved chunks for a given query to validate retrieval quality."""
    docs = retriever.invoke(query)
    print(f"Query: '{query}'")
    print(f"Retrieved {len(docs)} chunks\n")
    for i, doc in enumerate(docs[:top_k]):
        print(f"--- Chunk {i+1} ---")
        print(f"Source: {doc.metadata.get('source')} | Page: {doc.metadata.get('page', '?')}")
        print(doc.page_content[:400])
        print()

# Audit key queries used in the pipeline
audit_retrieval("Badre D'Esposito hierarchical abstraction definition")
audit_retrieval("PFC activation results Ito 2023 frontal cortex")

## 10. Discussion & Limitations

### What this pipeline does well
- Grounds LLM reasoning in retrieved text via explicit source citations
- Separates concerns across three agents with distinct epistemic roles
- The verifier is designed to flag unsupported inferences, not just confirm them

### Known limitations

| Limitation | Impact | Mitigation |
|---|---|---|
| MDTB (Ito 2023) reports whole-cortex topography, not per-task PFC subregion data | Agent 2 may have limited data for specific tasks | Add a targeted search for supplementary figures |
| bioRxiv scraping may fail silently in restricted network environments | Agent 3 may miss recent literature | Fall back to PubMed API or Semantic Scholar |
| Free-tier LLMs have variable instruction-following | Structured table outputs may be malformed | Add output parsers / Pydantic schemas |
| No re-ranking of retrieved chunks | Less relevant chunks may crowd out key passages | Add a cross-encoder reranker (e.g., `ms-marco-MiniLM`) |

### Potential extensions
- **Reranking:** Add a cross-encoder reranker between retrieval and generation
- **Structured outputs:** Use Pydantic models as `expected_output` schemas for parseable results
- **Citation tracing:** Log which chunks each agent used for full provenance
- **HyDE (Hypothetical Document Embeddings):** Generate a hypothetical answer first, embed it, then retrieve — improves recall for scientific queries